In [395]:
# %pip install thefuzz python-Levenshtein

# Original Data

In [396]:
import pandas as pd
df = pd.read_csv("C:\\Users\\usEr\\Desktop\\Project\\HIKARI\\Model\\SkinCAP\\skincap_v240623.csv")
df["disease"].value_counts()
df.columns

Index(['id', 'skincap_file_path', 'ori_file_path', 'disease', 'caption_zh',
       'caption_zh_polish', 'caption_zh_polish_en', 'remark', 'source',
       'skin_tone', 'malignant', 'fitzpatrick_scale', 'fitzpatrick_centaur',
       'nine_partition_label', 'three_partition_label', 'url', 'Vesicle',
       'Papule', 'Macule', 'Plaque', 'Abscess', 'Pustule', 'Bulla', 'Patch',
       'Nodule', 'Ulcer', 'Crust', 'Erosion', 'Excoriation', 'Atrophy',
       'Exudate', 'Purpura/Petechiae', 'Fissure', 'Induration', 'Xerosis',
       'Telangiectasia', 'Scale', 'Scar', 'Friable', 'Sclerosis',
       'Pedunculated', 'Exophytic/Fungating', 'Warty/Papillomatous',
       'Dome-shaped', 'Flat topped', 'Brown(Hyperpigmentation)', 'Translucent',
       'White(Hypopigmentation)', 'Purple', 'Yellow', 'Black', 'Erythema',
       'Comedo', 'Lichenification', 'Blue', 'Umbilicated', 'Poikiloderma',
       'Salmon', 'Wheal', 'Acuminate', 'Burrow', 'Gray', 'Pigmented', 'Cyst',
       'Do not consider this image

In [397]:
len(df['disease'].unique())

187

In [398]:
# main
df["disease"].value_counts()[:10]

disease
psoriasis                  127
squamous cell carcinoma    121
melanocytic-nevi           119
lupus erythematosus         95
lichen planus               92
basal cell carcinoma        90
scleroderma                 81
photodermatoses             76
acne vulgaris               76
sarcoidosis                 75
Name: count, dtype: int64

In [399]:
# df[df["disease"] == "acquired autoimmune bullous diseaseherpes gestationis"]

# Fuzzy Match Logic

In [400]:
# use fuzzy match to group the same label but in difference format for df["disease"] and show value_counts
import pandas as pd
from thefuzz import process

# 1. Define a similarity threshold (0-100). 80-90 is usually the sweet spot.
threshold = 95

# 2. Get unique values to reduce redundant calculations
unique_diseases = df['disease'].unique()

# 3. Create a mapping dictionary
mapping = {}
processed = set()

for disease in unique_diseases:
    if disease in processed:
        continue
    
    # Find all matches in the unique list that meet the threshold
    matches = process.extract(disease, unique_diseases, limit=None)
    close_matches = [match[0] for match in matches if match[1] >= threshold]
    
    # Map all close matches to the first occurrence (the "standard" label)
    for match in close_matches:
        mapping[match] = disease
        processed.add(match)

# 4. Apply mapping and show counts
df['disease_cleaned'] = df['disease'].map(mapping)
print(df['disease_cleaned'].value_counts())

disease_cleaned
squamous-cell-carcinoma-in-situ                 166
basal-cell-carcinoma                            137
psoriasis                                       127
melanocytic-nevi                                119
lupus erythematosus                              95
                                               ... 
leukemia-cutis                                    1
blastic-plasmacytoid-dendritic-cell-neoplasm      1
neuroma                                           1
acne-cystic                                       1
morphea                                           1
Name: count, Length: 175, dtype: int64


In [401]:
print(df['disease_cleaned'].value_counts().head(10))

disease_cleaned
squamous-cell-carcinoma-in-situ    166
basal-cell-carcinoma               137
psoriasis                          127
melanocytic-nevi                   119
lupus erythematosus                 95
lichen planus                       92
scleroderma                         81
photodermatoses                     76
acne vulgaris                       76
sarcoidosis                         75
Name: count, dtype: int64


### Show Match Group

In [402]:
# Grouping the original names by their new cleaned labels
grouped_data = df.groupby('disease_cleaned')['disease'].unique()

for clean_label, original_variants in grouped_data.items():
    print(f"Standard Label: {clean_label}")
    print(f"  -> Combined variants: {list(original_variants)}\n")

Standard Label: abrasions-ulcerations-and-physical-injuries
  -> Combined variants: ['abrasions-ulcerations-and-physical-injuries']

Standard Label: abscess
  -> Combined variants: ['abscess']

Standard Label: acanthosis nigricans
  -> Combined variants: ['acanthosis nigricans']

Standard Label: acne
  -> Combined variants: ['acne']

Standard Label: acne vulgaris
  -> Combined variants: ['acne vulgaris']

Standard Label: acne-cystic
  -> Combined variants: ['acne-cystic']

Standard Label: acquired autoimmune bullous diseaseherpes gestationis
  -> Combined variants: ['acquired autoimmune bullous diseaseherpes gestationis']

Standard Label: acquired-digital-fibrokeratoma
  -> Combined variants: ['acquired-digital-fibrokeratoma']

Standard Label: acral-melanotic-macule
  -> Combined variants: ['acral-melanotic-macule']

Standard Label: acrochordon
  -> Combined variants: ['acrochordon']

Standard Label: acrodermatitis enteropathica
  -> Combined variants: ['acrodermatitis enteropathica']


## Medical Logic Grouped

In [403]:
import csv
from collections import Counter, defaultdict

CSV_FILE = "C:\\Users\\usEr\\Desktop\\Project\\HIKARI\\Model\\SkinCAP\\skincap_v240623.csv"
DISEASE_COLUMN = "disease"


def norm(text: str) -> str:
    if not text:
        return ""
    return text.lower().replace("-", " ").strip()


# =========================
# Disease Groups (Expert-defined)
# =========================
DISEASE_GROUPS = {
    "Benign Tumors, Nevi & Cysts": {
        norm(d) for d in [
            "melanocytic nevi","seborrheic keratosis","granuloma annulare",
            "pyogenic granuloma","epidermal cyst","dermatofibroma",
            "juvenile xanthogranuloma","keloid","pilar cyst","milia",
            "syringoma","epidermal nevus","halo nevus","lymphangioma",
            "mucous cyst","porokeratosis of mibelli","fordyce spots",
            "acrochordon","nevus sebaceous of jadassohn","congenital nevus",
            "dysplastic nevus","xanthomas","nevocytic nevus",
            "seborrheic keratosis irritated","neurofibroma","angioma",
            "becker nevus","pilomatricoma","eccrine poroma",
            "granuloma pyogenic","lipoma","blue nevus",
            "nevus lipomatosus superficialis","inverted follicular keratosis",
            "trichilemmoma","benign keratosis","arteriovenous hemangioma",
            "foreign body granuloma","acquired digital fibrokeratoma",
            "syringocystadenoma papilliferum","trichofolliculoma",
            "xanthogranuloma","fibrous papule",
            "focal acral hyperkeratosis",
            "atypical spindle cell nevus of reed",
            "verruciform xanthoma","neuroma",
            "pigmented spindle cell nevus of reed",
            "glomangioma","lichenoid keratosis",
            "reactive lymphoid hyperplasia","chondroid syringoma"
        ]
    },

    "Inflammatory & Autoimmune Diseases": {
        norm(d) for d in [
            "psoriasis","lupus erythematosus","lichen planus","scleroderma",
            "photodermatoses","sarcoidosis","allergic contact dermatitis",
            "neutrophilic dermatoses","erythema multiforme",
            "prurigo nodularis","eczema","pityriasis rosea",
            "dermatomyositis","seborrheic dermatitis","urticaria",
            "lupus subacute","necrobiosis lipoidica",
            "erythema annulare centrifigum","cheilitis","lichen simplex",
            "dyshidrotic eczema","erythema nodosum","stasis edema",
            "factitial dermatitis","pustular psoriasis","neurodermatitis",
            "pityriasis lichenoides chronica",
            "erythema elevatum diutinum","neurotic excoriations",
            "behcets disease","eczema spongiotic dermatitis","morphea",
            "acquired autoimmune bullous diseaseherpes gestationis"
        ]
    },

    "Malignant Skin Tumors": {
        norm(d) for d in [
            "squamous cell carcinoma","basal cell carcinoma",
            "mycosis fungoides","melanoma","kaposi sarcoma",
            "squamous cell carcinoma in situ","lentigo maligna",
            "malignant melanoma","superficial spreading melanoma ssm",
            "solid cystic basal cell carcinoma",
            "basal cell carcinoma morpheiform",
            "squamous cell carcinoma keratoacanthoma",
            "melanoma acral lentiginous","basal cell carcinoma nodular",
            "melanoma in situ","metastatic carcinoma",
            "basal cell carcinoma superficial",
            "subcutaneous t cell lymphoma","nodular melanoma (nm)",
            "leukemia cutis","sebaceous carcinoma",
            "blastic plasmacytoid dendritic cell neoplasm"
        ]
    },

    "Infections & Infestations": {
        norm(d) for d in [
            "scabies","verruca vulgaris","nematode infection",
            "lyme disease","pediculosis lids","tick bite",
            "tungiasis","paronychia","myiasis","molluscum contagiosum",
            "onychomycosis","condyloma accuminatum","wart",
            "coccidioidomycosis","tinea pedis","abscess"
        ]
    },

    "Genetic & Systemic Disorders": {
        norm(d) for d in [
            "hailey hailey disease","porphyria","tuberous sclerosis",
            "urticaria pigmentosa","dariers disease",
            "neurofibromatosis","acrodermatitis enteropathica",
            "incontinentia pigmenti","xeroderma pigmentosum",
            "lichen amyloidosis","ehlers danlos syndrome","mucinosis",
            "calcinosis cutis","langerhans cell histiocytosis",
            "ichthyosis vulgaris"
        ]
    },

    "Acne & Follicular Disorders": {
        norm(d) for d in [
            "acne vulgaris","folliculitis","pityriasis rubra pilaris",
            "acne","rosacea","perioral dermatitis","rhinophyma",
            "keratosis pilaris","hidradenitis","acne cystic"
        ]
    },

    "Pre-cancerous & Sun Damage": {
        norm(d) for d in [
            "actinic keratosis","porokeratosis actinic",
            "disseminated actinic porokeratosis",
            "sun damaged skin","solar lentigo"
        ]
    },

    "Drug Reactions": {
        norm(d) for d in [
            "drug eruption","stevens johnson syndrome",
            "fixed eruptions","drug induced pigmentary changes"
        ]
    },

    "Pigmentary Disorders": {
        norm(d) for d in [
            "vitiligo","hyperpigmentation"
        ]
    },

    "Others": {
        norm(d) for d in [
            "telangiectases","papilomatosis confluentes and reticulate",
            "striae","scleromyxedema","aplasia cutis","port wine stain",
            "acanthosis nigricans","naevus comedonicus",
            "epidermolysis bullosa","livedo reticularis",
            "abrasions ulcerations and physical injuries","scar",
            "graft vs host disease","lymphocytic infiltrations",
            "angioleiomyoma","hematoma","cellular neurothekeoma",
            "clear cell acanthoma","acral melanotic macule"
        ]
    }
}


def main():
    group_counter = Counter()
    unmapped = Counter()
    total_rows = 0

    with open(CSV_FILE, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total_rows += 1
            disease = norm(row.get(DISEASE_COLUMN, ""))

            found = False
            for group, diseases in DISEASE_GROUPS.items():
                if disease in diseases:
                    group_counter[group] += 1
                    found = True
                    break

            if not found:
                unmapped[disease] += 1

    # ===== OUTPUT =====
    print("=" * 50)
    print("GROUPED DISEASE COUNT SUMMARY")
    print("=" * 50)
    print(f"Total rows in CSV : {total_rows}")
    print("-" * 50)

    total_grouped = 0
    for g, c in group_counter.most_common():
        total_grouped += c
        print(f"{g:<35} {c}")

    print("-" * 50)
    print(f"Total grouped samples : {total_grouped}")

    if unmapped:
        print("\n⚠️ Unmapped diseases:")
        for d, c in unmapped.most_common():
            print(f"{d} : {c}")

    print("=" * 50)


if __name__ == "__main__":
    main()


GROUPED DISEASE COUNT SUMMARY
Total rows in CSV : 4000
--------------------------------------------------
Inflammatory & Autoimmune Diseases  1228
Benign Tumors, Nevi & Cysts         822
Malignant Skin Tumors               581
Genetic & Systemic Disorders        326
Acne & Follicular Disorders         321
Infections & Infestations           290
Others                              184
Drug Reactions                      110
Pre-cancerous & Sun Damage          97
Pigmentary Disorders                41
--------------------------------------------------
Total grouped samples : 4000


In [404]:
# Create a reverse mapping: { 'psoriasis': 'Inflammatory & Autoimmune Diseases', ... }
reverse_mapping = {}
for group_name, labels in DISEASE_GROUPS.items():
    for label in labels:
        reverse_mapping[label] = group_name

# 1. Standardize your column first (using your norm function logic)
df['disease_norm'] = df['disease'].apply(norm)

# 2. Map the High-Level Group
df['disease_group'] = df['disease_norm'].map(reverse_mapping).fillna("Uncategorized")

In [405]:
group_counts = df['disease_group'].value_counts()
print("--- THE GROUP LEVEL ---")
print(group_counts)

--- THE GROUP LEVEL ---
disease_group
Inflammatory & Autoimmune Diseases    1228
Benign Tumors, Nevi & Cysts            822
Malignant Skin Tumors                  581
Genetic & Systemic Disorders           326
Acne & Follicular Disorders            321
Infections & Infestations              290
Others                                 184
Drug Reactions                         110
Pre-cancerous & Sun Damage              97
Pigmentary Disorders                    41
Name: count, dtype: int64


In [406]:
# Show value counts of sub-items nested under their parent groups
sub_item_counts = df.groupby(['disease_group', 'disease_norm']).size().sort_values(ascending=False)

print("--- THE SUB-ITEM LEVEL (Top 10) ---")
print(sub_item_counts)

--- THE SUB-ITEM LEVEL (Top 10) ---
disease_group                       disease_norm                
Malignant Skin Tumors               squamous cell carcinoma         138
                                    basal cell carcinoma            131
Inflammatory & Autoimmune Diseases  psoriasis                       127
Benign Tumors, Nevi & Cysts         melanocytic nevi                119
Inflammatory & Autoimmune Diseases  lupus erythematosus              95
                                                                   ... 
Others                              cellular neurothekeoma            1
                                    acral melanotic macule            1
Malignant Skin Tumors               subcutaneous t cell lymphoma      1
Others                              angioleiomyoma                    1
                                    lymphocytic infiltrations         1
Length: 178, dtype: int64


In [407]:
def inspect_group(group_name):
    mask = df['disease_group'] == group_name
    print(f"\nBreakdown for: {group_name}")
    print(df[mask]['disease_norm'].value_counts())

# Example usage:
inspect_group("Malignant Skin Tumors")


Breakdown for: Malignant Skin Tumors
disease_norm
squamous cell carcinoma                         138
basal cell carcinoma                            131
mycosis fungoides                                68
melanoma                                         57
kaposi sarcoma                                   30
squamous cell carcinoma in situ                  28
lentigo maligna                                  23
malignant melanoma                               21
superficial spreading melanoma ssm               18
solid cystic basal cell carcinoma                15
basal cell carcinoma morpheiform                 14
squamous cell carcinoma keratoacanthoma           8
melanoma acral lentiginous                        7
basal cell carcinoma nodular                      6
melanoma in situ                                  5
metastatic carcinoma                              5
basal cell carcinoma superficial                  2
subcutaneous t cell lymphoma                      1
nodular melan

# Integrate Fuzzy with Medical Logic

In [408]:
import pandas as pd
from thefuzz import process

# --- STEP 0: Normalization Function ---
def norm(text):
    if not isinstance(text, str): return text
    return text.lower().strip().replace("-", " ")

# --- STEP 1: Fuzzy Consolidation (The Automate Step) ---
def fuzzy_cleanup(df_column, threshold=90):
    unique_vals = df_column.unique()
    mapping = {}
    processed = set()

    for val in unique_vals:
        if val in processed:
            continue
        # Find matches
        matches = process.extract(val, unique_vals, limit=None)
        close_matches = [m[0] for m in matches if m[1] >= threshold]
        
        # Map all variants to the FIRST one found
        for match in close_matches:
            mapping[match] = val
            processed.add(match)
    return mapping

# Apply Normalization
df['disease_norm'] = df['disease'].apply(norm)

# Generate Fuzzy Mapping and Apply it
fuzzy_map = fuzzy_cleanup(df['disease_norm'], threshold=91)
df['disease_fuzzy'] = df['disease_norm'].map(fuzzy_map)

# --- STEP 2: Medical Grouping (The Logic Step) ---
# Create reverse lookup from your DISEASE_GROUPS
reverse_group_map = {}
for group, labels in DISEASE_GROUPS.items():
    for label in labels:
        # We use norm() here to ensure the dictionary keys match the fuzzy data
        reverse_group_map[norm(label)] = group

# Map to High Level Groups
df['high_level_group'] = df['disease_fuzzy'].map(reverse_group_map).fillna("Others")

# --- STEP 3: Show the Results ---
print("--- TOP 10 SUB-ITEMS (After Fuzzy) ---")
print(df['disease_fuzzy'].value_counts().head(10))

print("\n--- GROUP LEVEL COUNTS ---")
print(df['high_level_group'].value_counts())

--- TOP 10 SUB-ITEMS (After Fuzzy) ---
disease_fuzzy
squamous cell carcinoma in situ    166
basal cell carcinoma               137
psoriasis                          127
melanocytic nevi                   119
lupus erythematosus                 95
lichen planus                       92
scleroderma                         81
photodermatoses                     76
acne vulgaris                       76
sarcoidosis                         75
Name: count, dtype: int64

--- GROUP LEVEL COUNTS ---
high_level_group
Inflammatory & Autoimmune Diseases    1228
Benign Tumors, Nevi & Cysts            822
Malignant Skin Tumors                  581
Genetic & Systemic Disorders           326
Acne & Follicular Disorders            321
Infections & Infestations              290
Others                                 184
Drug Reactions                         110
Pre-cancerous & Sun Damage              97
Pigmentary Disorders                    41
Name: count, dtype: int64


In [409]:
# Find cases where the original normalized name is different from the fuzzy name
diff_df = df[df['disease_norm'] != df['disease_fuzzy']][['disease_norm', 'disease_fuzzy']].drop_duplicates()

print(f"Total labels consolidated by Fuzzy: {len(diff_df)}")
print("\nExamples of consolidations:")
print(diff_df.head(10))

Total labels consolidated by Fuzzy: 3

Examples of consolidations:
                     disease_norm                    disease_fuzzy
4         squamous cell carcinoma  squamous cell carcinoma in situ
69   basal cell carcinoma nodular             basal cell carcinoma
716            granuloma pyogenic               pyogenic granuloma


In [410]:
# 1. Calculate counts for Original (Manual)
orig_counts = df['disease_norm'].map(reverse_group_map).fillna("Others").value_counts()

# 2. Calculate counts for Fuzzy + Manual
fuzzy_counts = df['high_level_group'].value_counts()

# 3. Combine into a Diff Table
diff_report = pd.DataFrame({
    'Manual_Only': orig_counts,
    'Fuzzy_Integrated': fuzzy_counts
}).fillna(0)

diff_report['Change'] = diff_report['Fuzzy_Integrated'] - diff_report['Manual_Only']
print("--- WHOLE GROUP DIFF REPORT ---")
print(diff_report.sort_values(by='Fuzzy_Integrated', ascending=False))

--- WHOLE GROUP DIFF REPORT ---
                                    Manual_Only  Fuzzy_Integrated  Change
Inflammatory & Autoimmune Diseases         1228              1228       0
Benign Tumors, Nevi & Cysts                 822               822       0
Malignant Skin Tumors                       581               581       0
Genetic & Systemic Disorders                326               326       0
Acne & Follicular Disorders                 321               321       0
Infections & Infestations                   290               290       0
Others                                      184               184       0
Drug Reactions                              110               110       0
Pre-cancerous & Sun Damage                   97                97       0
Pigmentary Disorders                         41                41       0


In [411]:
# 1. Compare the mappings for every unique disease label
analysis = df[['disease_norm', 'disease_fuzzy', 'high_level_group']].drop_duplicates()

# 2. Identify the labels where the Fuzzy Master differs from the original
# Or where the Fuzzy logic found a group that the manual norm-match missed
analysis['orig_group'] = analysis['disease_norm'].map(reverse_group_map).fillna("Others")

# Filter for changes
changed_labels = analysis[analysis['orig_group'] != analysis['high_level_group']]

print(f"--- DISEASES AFFECTED BY FUZZY RECLASSIFICATION (Total: {len(changed_labels)}) ---")
if not changed_labels.empty:
    print(changed_labels[['disease_norm', 'orig_group', 'high_level_group', 'disease_fuzzy']])
else:
    print("No changes found. Your Manual Dictionary was already perfect for this threshold!")

--- DISEASES AFFECTED BY FUZZY RECLASSIFICATION (Total: 0) ---
No changes found. Your Manual Dictionary was already perfect for this threshold!


In [412]:
# 1. Create a Helper DataFrame to compare
comparison_df = df[['disease_norm', 'disease_fuzzy', 'high_level_group']].drop_duplicates()

# 2. Count unique words per group: Original (Manual)
orig_word_count = comparison_df.groupby('high_level_group')['disease_norm'].nunique()

# 3. Count unique words per group: After Fuzzy (Automated)
fuzzy_word_count = comparison_df.groupby('high_level_group')['disease_fuzzy'].nunique()

# 4. Create the Diff Report
word_diff_report = pd.DataFrame({
    'Unique Words (Manual)': orig_word_count,
    'Unique Words (Fuzzy)': fuzzy_word_count
})

word_diff_report['Words Removed (Noise)'] = word_diff_report['Unique Words (Manual)'] - word_diff_report['Unique Words (Fuzzy)']

print("--- WORD COUNT DIFF PER GROUP ---")
print(word_diff_report.sort_values(by='Words Removed (Noise)', ascending=False))

# Show the total reduction
total_before = word_diff_report['Unique Words (Manual)'].sum()
total_after = word_diff_report['Unique Words (Fuzzy)'].sum()
print(f"\nTotal Unique Labels: {total_before} -> {total_after} (Reduced {total_before - total_after} labels)")

--- WORD COUNT DIFF PER GROUP ---
                                    Unique Words (Manual)  \
high_level_group                                            
Malignant Skin Tumors                                  22   
Benign Tumors, Nevi & Cysts                            52   
Acne & Follicular Disorders                            10   
Drug Reactions                                          4   
Infections & Infestations                              16   
Genetic & Systemic Disorders                           15   
Inflammatory & Autoimmune Diseases                     33   
Others                                                 19   
Pigmentary Disorders                                    2   
Pre-cancerous & Sun Damage                              5   

                                    Unique Words (Fuzzy)  \
high_level_group                                           
Malignant Skin Tumors                                 20   
Benign Tumors, Nevi & Cysts                          